# Geometry Working Memory

https://www.biorxiv.org/content/10.64898/2026.08.31.748237v1

In [ ]:
# libraries

import os
import numpy as np
import pickle
import scipy


In [ ]:
# functions

def compute_pca(X):

    """
    Compute PCA through eigendecomposition of covariance matrix
    pc_scores[:,0:k]: Leading k pc_scores
    eigvecs[:,0:k]:   Leading k eigengectors

    """

    # Step 1: Center the data (subtract the mean of each feature)
    X_mean = np.nanmean(X, axis=0)
    X_centered = X - X_mean

    # Step 2: Compute the covariance matrix
    cov_matrix = np.cov(X_centered, rowvar=False)

    # Step 3: Eigenvalue decomposition using eigh
    eigvals, eigvecs = scipy.linalg.eigh(cov_matrix)

    # Sort the eigenvectors by eigenvalues (descending order)
    sorted_indices = np.argsort(eigvals)[::-1]
    eigvals = eigvals[sorted_indices]
    eigvecs = eigvecs[:, sorted_indices]

    # Step 4: Compute principal component scores
    pc_scores = np.dot(X_centered, eigvecs)

    return eigvecs, eigvals, pc_scores


def compute_pa(eigvecs1, eigvecs2, k_start, k_end):
    """
    Compute Principal Angles between two subspaces defined by the eigenvectors.
    Returns a vector of angles, as many as the dimensionality of subspaces,
    defined as k_end - k_start

    Inputs:
        - eigvecs1/eigvecs2: eigvecs of subspaces 1 and 2 (with all PCs in columns)
        - k_start: first component to consider (starting from 0)
        - k_end: last component to consider (starting from 0)
            e.g., k_start, k_end = 0, 2 # will subset three components [0 1 2]

    Outputs:
        - angles: all principal angles

    """
    ### Define subspaces
    
    eigvecs_subspace1 = eigvecs1[:, k_start:(k_end + 1)]
    eigvecs_subspace2 = eigvecs2[:, k_start:(k_end + 1)]

    ### Principal Angle between subspaces

    # Compute the correlation matrix C = U1.T @ U2
    C = np.dot(eigvecs_subspace1.T, eigvecs_subspace2)

    # Perform Singular Value Decomposition (SVD)
    _, singular_values, _ = np.linalg.svd(C)

    # Compute the principal angles (in radians)
    angles = np.arccos(np.clip(singular_values, -1, 1))

    # Compute angles (in degrees)
    angles = np.degrees(angles)

    return angles


def compute_vaf(eigvecs1, eigvecs2, pc_scores1, pc_scores2, k_start, k_end):
    """
    Compute variance accounted for between two subspaces defined by eigenvectors 
    and its data projection (pc_scores) of dimensionality k_end - k_start.

    Inputs:
        - eigvecs1/eigvecs2: eigvecs of subspaces 1 and 2 (with all PCs in columns)
        - pc_scores1/pc_scores2: pc_scores of subspaces 1 and 2 (with all PCs in columns)
        - k_start: first component to consider (starting from 0)
        - k_end: last component to consider (starting from 0)
            e.g., k_start, k_end = 0, 2 # will subset three components [0 1 2]

    Outputs:
        - vaf: variance accounted for

    """

    ### Define subspaces
    
    pc_scores_subspace1 = pc_scores1[:, k_start:(k_end + 1)]
    pc_scores_subspace2 = pc_scores2[:, k_start:(k_end + 1)]

    eigvecs_subspace1 = eigvecs1[:, k_start:(k_end + 1)]
    eigvecs_subspace2 = eigvecs2[:, k_start:(k_end + 1)]

    ### Variance Accounted For (VAF)

    ## VAF 1-to-2
    numerator_1to2 = np.linalg.norm(eigvecs_subspace2 @ eigvecs_subspace2.T @ eigvecs_subspace1 @ pc_scores_subspace1.T, 'fro')
    denominator_1to2 = np.linalg.norm(eigvecs_subspace1 @ pc_scores_subspace1.T, 'fro')
    vaf_1to2 = (numerator_1to2 / denominator_1to2) ** 2

    ## VAF 2-to-1
    numerator_2to1 = np.linalg.norm(eigvecs_subspace1 @ eigvecs_subspace1.T @ eigvecs_subspace2 @ pc_scores_subspace2.T, 'fro')
    denominator_2to1 = np.linalg.norm(eigvecs_subspace2 @ pc_scores_subspace2.T, 'fro')
    vaf_2to1 = (numerator_2to1 / denominator_2to1) ** 2

    ## Average VAF
    vaf = (vaf_1to2 + vaf_2to1) / 2

    return vaf



def compute_vaf_best_fitting_plane(pc_scores1, pc_scores2, k_start, k_end):
    """
    
    Perform PCA on subspaces defined by pc_scores to get the best fitting plane (2 leading eigenvectors)
    and then compute VAF between subspaces planes.
    
    Inputs:
        - eigvecs1/eigvecs2: eigvecs of subspaces 1 and 2 (with all PCs in columns)
        - pc_scores1/pc_scores2: pc_scores of subspaces 1 and 2 (with all PCs in columns)
        - k_start: first component to consider (starting from 0)
        - k_end: last component to consider (starting from 0)
            e.g., k_start, k_end = 0, 2 # will subset three components [0 1 2]

    Outputs:
        - vaf: variance accounted for

    """

    ### Define subspaces with k_start:k_end components
    
    pc_scores_subspace1 = pc_scores1[:, k_start:(k_end + 1)]
    pc_scores_subspace2 = pc_scores2[:, k_start:(k_end + 1)]

    ### best fitting plane
    
    if k_end - k_start > 1:

        eigvecs_subspace1, _, pc_scores_subspace1 = compute_pca(pc_scores_subspace1)
        eigvecs_subspace2, _, pc_scores_subspace2 = compute_pca(pc_scores_subspace2)

        eigvecs_subspace1 = eigvecs_subspace1[:,0:2].copy()
        eigvecs_subspace2 = eigvecs_subspace2[:,0:2].copy()
        pc_scores_subspace1 = pc_scores_subspace1[:,0:2].copy()
        pc_scores_subspace2 = pc_scores_subspace2[:,0:2].copy()

    ### Variance Accounted For (VAF)

    ## VAF 1-to-2
    numerator_1to2 = np.linalg.norm(eigvecs_subspace2 @ eigvecs_subspace2.T @ eigvecs_subspace1 @ pc_scores_subspace1.T, 'fro')
    denominator_1to2 = np.linalg.norm(eigvecs_subspace1 @ pc_scores_subspace1.T, 'fro')
    vaf_1to2 = (numerator_1to2 / denominator_1to2) ** 2

    ## VAF 2-to-1
    numerator_2to1 = np.linalg.norm(eigvecs_subspace1 @ eigvecs_subspace1.T @ eigvecs_subspace2 @ pc_scores_subspace2.T, 'fro')
    denominator_2to1 = np.linalg.norm(eigvecs_subspace2 @ pc_scores_subspace2.T, 'fro')
    vaf_2to1 = (numerator_2to1 / denominator_2to1) ** 2

    ## Average VAF
    vaf = (vaf_1to2 + vaf_2to1) / 2

    return vaf

def compute_pa_best_fitting_plane(pc_scores1, pc_scores2, k_start, k_end):
    """

    Perform PCA on subspaces defined by pc_scores to get the best fitting plane (2 leading eigenvectors)
    and then compute principal angle between subspaces planes.

    Inputs:
        - eigvecs1/eigvecs2: eigvecs of subspaces 1 and 2 (with all PCs in columns)
        - pc_scores1/pc_scores2: pc_scores of subspaces 1 and 2 (with all PCs in columns)
        - k_start: first component to consider (starting from 0)
        - k_end: last component to consider (starting from 0)
            e.g., k_start, k_end = 0, 2 # will subset three components [0 1 2]

    Outputs:
        - angles: all principal angles

    """

    ### Define subspaces with k_start:k_end components
    
    pc_scores_subspace1 = pc_scores1[:, k_start:(k_end + 1)]
    pc_scores_subspace2 = pc_scores2[:, k_start:(k_end + 1)]

    ### best fitting plane
    
    if k_end - k_start > 1:

        eigvecs_subspace1, _, pc_scores_subspace1 = compute_pca(pc_scores_subspace1)
        eigvecs_subspace2, _, pc_scores_subspace2 = compute_pca(pc_scores_subspace2)

        eigvecs_subspace1 = eigvecs_subspace1[:,0:2].copy()
        eigvecs_subspace2 = eigvecs_subspace2[:,0:2].copy()
        pc_scores_subspace1 = pc_scores_subspace1[:,0:2].copy()
        pc_scores_subspace2 = pc_scores_subspace2[:,0:2].copy()

    ### Principal Angle between subspaces

    # Compute the correlation matrix C = U1.T @ U2
    C = np.dot(eigvecs_subspace1.T, eigvecs_subspace2)

    # Perform Singular Value Decomposition (SVD)
    _, singular_values, _ = np.linalg.svd(C)

    # Compute the principal angles (in radians)
    angles = np.arccos(np.clip(singular_values, -1, 1))

    # Compute angles (in degrees)
    angles = np.degrees(angles)

    return angles

def compute_vaf_best_fitting_plane(pc_scores1, pc_scores2, k_start, k_end):
    """
    
    Perform PCA on subspaces defined by pc_scores to get the best fitting plane (2 leading eigenvectors)
    and then compute VAF between subspaces planes.
    
    Inputs:
        - eigvecs1/eigvecs2: eigvecs of subspaces 1 and 2 (with all PCs in columns)
        - pc_scores1/pc_scores2: pc_scores of subspaces 1 and 2 (with all PCs in columns)
        - k_start: first component to consider (starting from 0)
        - k_end: last component to consider (starting from 0)
            e.g., k_start, k_end = 0, 2 # will subset three components [0 1 2]

    Outputs:
        - vaf: variance accounted for

    """

    ### Define subspaces with k_start:k_end components
    
    pc_scores_subspace1 = pc_scores1[:, k_start:(k_end + 1)]
    pc_scores_subspace2 = pc_scores2[:, k_start:(k_end + 1)]

    ### best fitting plane
    
    if k_end - k_start > 1:

        eigvecs_subspace1, _, pc_scores_subspace1 = compute_pca(pc_scores_subspace1)
        eigvecs_subspace2, _, pc_scores_subspace2 = compute_pca(pc_scores_subspace2)

        eigvecs_subspace1 = eigvecs_subspace1[:,0:2].copy()
        eigvecs_subspace2 = eigvecs_subspace2[:,0:2].copy()
        pc_scores_subspace1 = pc_scores_subspace1[:,0:2].copy()
        pc_scores_subspace2 = pc_scores_subspace2[:,0:2].copy()

    ### Variance Accounted For (VAF)

    ## VAF 1-to-2
    numerator_1to2 = np.linalg.norm(eigvecs_subspace2 @ eigvecs_subspace2.T @ eigvecs_subspace1 @ pc_scores_subspace1.T, 'fro')
    denominator_1to2 = np.linalg.norm(eigvecs_subspace1 @ pc_scores_subspace1.T, 'fro')
    vaf_1to2 = (numerator_1to2 / denominator_1to2) ** 2

    ## VAF 2-to-1
    numerator_2to1 = np.linalg.norm(eigvecs_subspace1 @ eigvecs_subspace1.T @ eigvecs_subspace2 @ pc_scores_subspace2.T, 'fro')
    denominator_2to1 = np.linalg.norm(eigvecs_subspace2 @ pc_scores_subspace2.T, 'fro')
    vaf_2to1 = (numerator_2to1 / denominator_2to1) ** 2

    ## Average VAF
    vaf = (vaf_1to2 + vaf_2to1) / 2

    return vaf



In [ ]:
### set paths and settings

path_root = /path_to_local'

# settings

subjects = [f'sub_{i:02d}' for i in range(1, 50)]

time_windows = ['encode', 'maint', 's2']
pca_aligned_folder = 'pca_aligned'
k_start, k_end = 0, 2       # 0,2 means leading three PCs
trial_types = ['correct_trials', 'incorrect_trials']

outputs_correct_trials = [
'control_2gratings_2polygons',                                  'control_1gratings_1polygons',
'update_2gratings_relevant_2polygons_nonrelevant',              'update_1gratings_relevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_relevant',              'update_1gratings_nonrelevant_1polygons_relevant',
'inhibition_2gratings_relevant_2polygons_nonrelevant',          'inhibition_1gratings_relevant_1polygons_nonrelevant',
'inhibition_2gratings_nonrelevant_2polygons_relevant',          'inhibition_1gratings_nonrelevant_1polygons_relevant',
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      'update_1gratings_nolongerrelevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',      'update_1gratings_nonrelevant_1polygons_nolongerrelevant',
]

outputs_incorrect_trials = [
'control_2gratings_2polygons',                                  
'update_2gratings_relevant_2polygons_nonrelevant',              
'update_2gratings_nonrelevant_2polygons_relevant',              
'inhibition_2gratings_relevant_2polygons_nonrelevant',          
'inhibition_2gratings_nonrelevant_2polygons_relevant',        
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',       
]


# Alignment - PA and VAF

In [23]:
for trial_type in trial_types:

    if trial_type == 'correct_trials':
        outputs = outputs_correct_trials
    elif trial_type == 'incorrect_trials':
        outputs = outputs_incorrect_trials

    for time_window in time_windows:

        if time_window == 'encode':

            segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

        elif time_window == 'maint':

            segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

        elif time_window == 's2':

            segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

        path_outputs = os.path.join(path_root, 'results', pca_aligned_folder, time_window + '_time_resolved', 'allsubjects', trial_type)
        if not os.path.isdir(path_outputs):
            os.makedirs(path_outputs)

        if not os.path.isfile(os.path.join(path_outputs, 'vaf_' + trial_type + '_' + time_window + '_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl')):

            pa_dict = {}
            vaf_dict = {}

            # loop over conditions (including stimulus load)
            for output_idx, output_i in enumerate(outputs):

                # empty matrix for outputs
                pa_matrix = np.empty((len(subjects), len(segments)))
                vaf_matrix = np.empty((len(subjects), len(segments)))

                for sub_idx, sub_i in enumerate(subjects):

                    # load file with pca outputs
                    file = os.path.join(path_root, 'results', pca_aligned_folder, time_window + '_time_resolved', sub_i, 'pca_' + trial_type + '.pkl')
                    with open(file, 'rb') as file:
                        data = pickle.load(file)

                    for time_idx, time_segment in enumerate(segments):

                        key = time_window + '_segment' + str(time_segment) + '_' + output_i

                        pc_scores = data['pc_scores_' + key]
                        pc_scores_orientation = pc_scores[0:4,k_start:(k_end+1)]
                        pc_scores_shape = pc_scores[4:7,k_start:(k_end+1)]

                        pa_matrix[sub_idx,time_idx] = np.max(compute_pa_best_fitting_plane(pc_scores_orientation, pc_scores_shape, k_start, k_end))
                        vaf_matrix[sub_idx,time_idx] = compute_vaf_best_fitting_plane(pc_scores_orientation, pc_scores_shape, k_start, k_end)

                # assign to dict
                pa_dict[output_i] = pa_matrix
                vaf_dict[output_i] = vaf_matrix

            # save            
            with open(os.path.join(path_outputs, 'pa_' + trial_type + '_' + time_window + '_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
                pickle.dump(pa_dict, file)

            with open(os.path.join(path_outputs, 'vaf_' + trial_type + '_' + time_window + '_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
                pickle.dump(vaf_dict, file)
